CELDA 1 — Instalación de librerías

In [1]:
!pip install fastapi uvicorn pyngrok pymongo dnspython nest-asyncio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.6 MB/s eta 0:00:00


CELDA 2 — Crear el archivo main.py

In [2]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pymongo import MongoClient
import os

app = FastAPI(
    title="API Ernesto Investing AI",
    description="API REST que expone datos de mercado y predicciones de IA desde MongoDB",
    version="1.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Conexión a MongoDB (la URI se inyecta como variable de entorno antes de levantar uvicorn)
MONGO_URI = os.environ["MONGO_URI"]
client = MongoClient(MONGO_URI)
db = client["ernesto_investing_ai"]

coleccion_precios = db["precios_ohlcv"]
coleccion_predicciones = db["predicciones"]
coleccion_metricas = db["metricas_modelos"]

TICKERS_VALIDOS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
MODELOS_RNN = ["LSTM", "BiLSTM", "GRU", "SimpleRNN"]


def limpiar_documento(doc):
    """Excluye _id y campos internos antes de devolver el JSON."""
    if doc is None:
        return None
    doc = dict(doc)
    doc.pop("_id", None)
    if "fecha_actualizacion" in doc:
        doc["fecha_actualizacion"] = doc["fecha_actualizacion"].isoformat()
    if "actualizado_en" in doc:
        doc["actualizado_en"] = doc["actualizado_en"].isoformat()
    return doc


def validar_ticker(ticker: str):
    if ticker not in TICKERS_VALIDOS:
        raise HTTPException(
            status_code=404,
            detail=f"Ticker '{ticker}' no válido. Tickers disponibles: {TICKERS_VALIDOS}"
        )


@app.get("/api/salud")
def salud():
    try:
        client.admin.command("ping")
        return {"estado": "ok", "mensaje": "API y MongoDB funcionando correctamente"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error de conexión a MongoDB: {e}")


@app.get("/api/mercado/{ticker}")
def mercado(ticker: str):
    validar_ticker(ticker)
    doc = coleccion_precios.find_one({"ticker": ticker})
    if doc is None:
        raise HTTPException(status_code=404, detail=f"No hay datos de mercado para {ticker}")
    return limpiar_documento(doc)


@app.get("/api/svc/{ticker}")
def svc(ticker: str):
    validar_ticker(ticker)
    prediccion = coleccion_predicciones.find_one({"ticker": ticker, "modelo": "SVC"})
    metricas = coleccion_metricas.find_one({"ticker": ticker, "modelo": "SVC"})

    if prediccion is None or metricas is None:
        raise HTTPException(status_code=404, detail=f"No hay resultados SVC para {ticker}")

    return {
        "prediccion": limpiar_documento(prediccion),
        "metricas": limpiar_documento(metricas)
    }


@app.get("/api/rnns/{ticker}")
def rnns(ticker: str):
    validar_ticker(ticker)

    resultados = {}
    for modelo in MODELOS_RNN:
        prediccion = coleccion_predicciones.find_one({"ticker": ticker, "modelo": modelo})
        metricas = coleccion_metricas.find_one({"ticker": ticker, "modelo": modelo})

        if prediccion is None or metricas is None:
            continue

        resultados[modelo] = {
            "prediccion": limpiar_documento(prediccion),
            "metricas": limpiar_documento(metricas)
        }

    if not resultados:
        raise HTTPException(status_code=404, detail=f"No hay resultados RNN para {ticker}")

    return resultados


@app.get("/api/lstm/{ticker}")
def lstm_regressor(ticker: str, horizonte: int = 30):
    validar_ticker(ticker)

    if horizonte not in [7, 14, 30, 60]:
        raise HTTPException(
            status_code=400,
            detail="Horizonte debe ser uno de: 7, 14, 30, 60"
        )

    prediccion = coleccion_predicciones.find_one({"ticker": ticker, "modelo": "LSTM_Regressor"})
    metricas = coleccion_metricas.find_one({"ticker": ticker, "modelo": "LSTM_Regressor"})

    if prediccion is None or metricas is None:
        raise HTTPException(status_code=404, detail=f"No hay resultados LSTM Regressor para {ticker}")

    prediccion_limpia = limpiar_documento(prediccion)

    return {
        "fechas_test": prediccion_limpia["fechas_test"],
        "precios_reales_test": prediccion_limpia["precios_reales_test"],
        "precios_predichos_test": prediccion_limpia["precios_predichos_test"],
        "prediccion_horizonte": prediccion_limpia["prediccion_futura"][str(horizonte)],
        "serie_futura_completa": prediccion_limpia["serie_futura_completa"][:horizonte],
        "metricas": limpiar_documento(metricas)
    }

Writing main.py


CELDA 3 — Preparar el entorno

In [3]:
import nest_asyncio
from google.colab import userdata
import os

nest_asyncio.apply()

# Inyectar el MONGO_URI como variable de entorno para que main.py lo lea
os.environ["MONGO_URI"] = userdata.get("MONGO_URI")

CELDA 4 — Configurar ngrok

In [6]:
from pyngrok import ngrok, conf

# Guarda tu authtoken de ngrok como Secret en Colab (nombre: NGROK_TOKEN)
# Se obtiene gratis en https://dashboard.ngrok.com/get-started/your-authtoken
conf.get_default().auth_token = userdata.get("NGROK_TOKEN")

# Cerrar túneles previos por si quedaron abiertos de una ejecución anterior
ngrok.kill()

CELDA 5 — Levantar el servidor

In [ ]:
import uvicorn
from main import app
import asyncio

public_url = ngrok.connect(8000)
url_limpia = public_url.public_url

print(f"🌐 API pública disponible en: {url_limpia}")
print(f"📄 Documentación Swagger: {url_limpia}/docs")
print(f"❤️  Health check: {url_limpia}/api/salud")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

🌐 API pública disponible en: https://untracked-gradient-poker.ngrok-free.dev
📄 Documentación Swagger: https://untracked-gradient-poker.ngrok-free.dev/docs
❤️  Health check: https://untracked-gradient-poker.ngrok-free.dev/api/salud


INFO:     Started server process [2341]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /api/salud HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /api/salud HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /api/salud HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2001:1388:1761:1768:d57d:914b:f5:3ca7:0 - "GET /openapi.jso